In [2]:
# 202 - Create SparkSession

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CustomerChurn") \
    .master("local[2]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [3]:
# 203 - Defining Manual Schema

from pyspark.sql.types import *

schema = StructType([
    StructField("customerID", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("SeniorCitizen", IntegerType(), True),
    StructField("Partner", StringType(), True),
    StructField("Dependents", StringType(), True),
    StructField("tenure", IntegerType(), True),
    StructField("PhoneService", StringType(), True),
    StructField("MultipleLines", StringType(), True),
    StructField("InternetService", StringType(), True),
    StructField("OnlineSecurity", StringType(), True),
    StructField("OnlineBackup", StringType(), True),
    StructField("DeviceProtection", StringType(), True),
    StructField("TechSupport", StringType(), True),
    StructField("StreamingTV", StringType(), True),
    StructField("StreamingMovies", StringType(), True),
    StructField("Contract", StringType(), True),
    StructField("PaperlessBilling", StringType(), True),
    StructField("PaymentMethod", StringType(), True),
    StructField("MonthlyCharges", DoubleType(), True),
    StructField("TotalCharges", StringType(), True),
    StructField("Churn", StringType(), True)
])

In [4]:
# 204 - read csv

df = spark.read.csv(
    "../../dataset/telco_customer.csv",
    header=True,
    schema = schema
)

In [13]:
import pandas as pd

pdf = pd.read_csv("../../dataset/telco_customer.csv")

In [14]:
print(df.printSchema())
print(pdf.dtypes)

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)
 |-- churn_encoded: integer (nullable = false)

None
customerID              str
g

In [8]:
# 205 - fix totalCharges

from pyspark.sql import functions as F

df = df.withColumn(
    "TotalCharges",
    F.regexp_replace(F.col("TotalCharges"), r"^\s*$", "0")
    .cast("double")
)

In [9]:
# 206 - encode churn

df = df.withColumn(
    "churn_encoded",
    F.when(F.col("Churn") == "Yes", 1).otherwise(0)
)

In [10]:
# 207 - group by contract

contract_summary = df.groupBy("Contract").agg(
    F.count("*").alias("customer_count"),
    F.avg("churn_encoded").alias("churn_rate")
)

contract_summary.show()

+--------------+--------------+-------------------+
|      Contract|customer_count|         churn_rate|
+--------------+--------------+-------------------+
|Month-to-month|          3875| 0.4270967741935484|
|      One year|          1473|0.11269517990495587|
|      Two year|          1695|0.02831858407079646|
+--------------+--------------+-------------------+



In [11]:
# 208 - group by internetService

contract_summary = df.groupBy("Contract").agg(
    F.count("*").alias("customer_count"),
    F.avg("churn_encoded").alias("churn_rate")
)

contract_summary.show()

+--------------+--------------+-------------------+
|      Contract|customer_count|         churn_rate|
+--------------+--------------+-------------------+
|Month-to-month|          3875| 0.4270967741935484|
|      One year|          1473|0.11269517990495587|
|      Two year|          1695|0.02831858407079646|
+--------------+--------------+-------------------+



In [15]:
# 209 - cross validate

pandas_contract = pdf.groupby("Contract")["Churn"].apply(
    lambda x: (x == "Yes").mean()
)

In [24]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [27]:
print("Hadoop:", os.path.exists(r"C:\hadoop"))
print("winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))

Hadoop: True
winutils: True


In [45]:
# 210 - write parquet

output_dir = r"D:\Training Project\Project 1\dataset\spark_output\contract_summary"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "data.parquet")

contract_summary.toPandas().to_parquet(
    file_path,
    engine="fastparquet",
    index=False
)

C:\Users\madhan.kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [46]:
# 211 - read parquet and validate round-trip

read_back = spark.read.parquet(file_path)

read_back.printSchema()

print("Row count:", read_back.count())

root
 |-- Contract: string (nullable = true)
 |-- customer_count: long (nullable = true)
 |-- churn_rate: double (nullable = true)

Row count: 3
